# Sentiment Analysis - Kindle Reviews

## Workflow:

1. Preprocessing and Cleaning
2. Train Test Split
3. BOW, TFIDF, Word2Vec
4. Train ML Algorithms

### Load dfset

In [89]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/data/kindle_review.csv")

df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


#### Important columns / features for the sentiment analysis task - rating and reviewTest

In [90]:
df = df[['reviewText', 'rating']]

In [91]:
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [92]:
df.shape

(12000, 2)

Check for missing values

In [93]:
df.isnull().sum()

,0
reviewText,0
rating,0


In [94]:
df['rating'].unique()

array([3, 5, 4, 2, 1])

Check df Balaced or not

In [39]:
df['rating'].value_counts()

,count
rating,
5,3000
4,3000
3,2000
2,2000
1,2000


Revising the rating to analyse sentiment

- rating > 3 = Positive
- rating < 3 = Negative


Positive review is 1 and negative review is 0

In [40]:
df['rating'] = df['rating'].apply(lambda x:0 if x < 3 else 1)

In [41]:
df.rating.unique()

array([1, 0])

In [42]:
df.rating.value_counts()

,count
rating,
1,8000
0,4000


## Text Preprocessing

Lower case the review text

In [43]:
df['reviewText'].str.lower()

,reviewText
0,"jace rankin may be short, but he's nothing to ..."
1,great short read. i didn't want to put it dow...
2,i'll start by saying this is the first of four...
3,aggie is angela lansbury who carries pocketboo...
4,i did not expect this type of book to be in li...
...,...
11995,valentine cupid is a vampire- jena and ian ano...
11996,i have read all seven books in this series. ap...
11997,this book really just wasn't my cuppa. the si...
11998,"tried to use it to charge my kindle, it didn't..."


Clean the review Text

- remove special characters

In [47]:
import nltk

# Download NLTK resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [49]:
!pip install beautifulsoup4

In [51]:
from bs4 import BeautifulSoup
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer


# Remove Special Characters
df.reviewText = df.reviewText.apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]', '', x))

# Remove stopwords and join
df.reviewText = df.reviewText.apply(lambda x: ' '.join([word for word in x.split() if word not in (stopwords.words('english'))]))

# Remove URL
url_pattern = r'\b(?:https?://|www\.)?[a-zA-Z0-9-]+(?:\.[a-zA-Z0-9-]+)+(?:/[^\s]*)?'
df.reviewText = df.reviewText.apply(lambda x: re.sub(url_pattern, '', x))

# Remove HTML Tags
df.reviewText = df.reviewText.apply(lambda x: BeautifulSoup(x, 'lxml').get_text())

# Remove any additional spaces
df.reviewText = df.reviewText.apply(lambda x: ' '.join(x.split()))

In [52]:
df.head()

,reviewText,rating
0,Jace Rankin may short hes nothing mess man hau...,1
1,Great short read I didnt want put I read one s...,1
2,Ill start saying first four books I wasnt expe...,1
3,Aggie Angela Lansbury carries pocketbooks inst...,1
4,I expect type book library pleased find price ...,1


Lemmatization

In [53]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

In [54]:
def lemmatize_word(text):
  return [lemmatizer.lemmatize(word) for word in text.split()]

In [55]:
df.reviewText = df['reviewText'].apply(lambda x: lemmatize_word(x))

In [56]:
df.head()

,reviewText,rating
0,"[Jace, Rankin, may, short, he, nothing, mess, ...",1
1,"[Great, short, read, I, didnt, want, put, I, r...",1
2,"[Ill, start, saying, first, four, book, I, was...",1
3,"[Aggie, Angela, Lansbury, carry, pocketbook, i...",1
4,"[I, expect, type, book, library, pleased, find...",1


In [57]:
df.rating.unique()

array([1, 0])

## Train Test Split

In [61]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.reviewText, df.rating, test_size=0.2, random_state=42)

In [65]:
X_train = X_train.apply(lambda x: ' '.join(x))
X_test = X_test.apply(lambda x: ' '.join(x))

In [105]:
X_train

,reviewText
9182,I looking forward book Then came double space ...
11091,We already owned book My spouse forgot already...
6428,Cool forgot I request rate came make mine unre...
288,This short short story Its basically scene par...
2626,All Secret Service agent secrests even longer ...
...,...
11964,I downloaded book reading review I usually Aft...
5191,This far one THE hottest book Ive ever gotten ...
5390,Even though book free I reservation Based majo...
860,A little mushy 34must take care woman folk34 T...


In [67]:
X_train.head()

,reviewText
9182,I looking forward book Then came double space ...
11091,We already owned book My spouse forgot already...
6428,Cool forgot I request rate came make mine unre...
288,This short short story Its basically scene par...
2626,All Secret Service agent secrests even longer ...


# Applying Bag of Words (BOW)

In [69]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer()

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

# Applying TF-IDF

In [71]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Naive bayes Model Training and Prediction

In [73]:
from sklearn.naive_bayes import GaussianNB

nb_bow = GaussianNB()
nb_tfidf = GaussianNB()


nb_bow.fit(X_train_bow.toarray(), y_train)
nb_tfidf.fit(X_train_tfidf.toarray(), y_train)

GaussianNB()

In [74]:
y_pred_bow = nb_bow.predict(X_test_bow.toarray())
y_pred_tfidf = nb_tfidf.predict(X_test_tfidf.toarray())

## Performance Metrics

In [79]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("BOW METRICS\n")
print("BOW Accuracy:\n", accuracy_score(y_test, y_pred_bow))
print("\nBOW Confusion Matrix:\n", confusion_matrix(y_test, y_pred_bow))
print("\nBOW Classification Report:\n", classification_report(y_test, y_pred_bow))


print("\n\nTFIDF METRICS")
print("\nTFIDF Accuracy:\n", accuracy_score(y_test, y_pred_tfidf))
print("\nTFIDF Confusion Matrix:\n", confusion_matrix(y_test, y_pred_tfidf))
print("\nTFIDF Classification Report:\n", classification_report(y_test, y_pred_tfidf))

BOW METRICS

BOW Accuracy:
 0.5783333333333334

BOW Confusion Matrix:
 [[492 311]
 [701 896]]

BOW Classification Report:
               precision    recall  f1-score   support

           0       0.41      0.61      0.49       803
           1       0.74      0.56      0.64      1597

    accuracy                           0.58      2400
   macro avg       0.58      0.59      0.57      2400
weighted avg       0.63      0.58      0.59      2400



TFIDF METRICS

TFIDF Accuracy:
 0.5791666666666667

TFIDF Confusion Matrix:
 [[477 326]
 [684 913]]

TFIDF Classification Report:
               precision    recall  f1-score   support

           0       0.41      0.59      0.49       803
           1       0.74      0.57      0.64      1597

    accuracy                           0.58      2400
   macro avg       0.57      0.58      0.56      2400
weighted avg       0.63      0.58      0.59      2400



# Applying Word2Vec and AvgWord2Vec

In [81]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 29.6 MB/s  0:00:00


In [104]:
import gensim
from gensim.utils import simple_preprocess

#### Extracting Words per sentence

In [106]:
X_train_words = [simple_preprocess(text) for text in X_train]
X_test_words = [simple_preprocess(text) for text in X_test]

In [107]:
X_train_words

[['looking',
  'forward',
  'book',
  'then',
  'came',
  'double',
  'space',
  'every',
  'paragraph',
  'kindle',
  'edition',
  'since',
  'action',
  'move',
  'around',
  'formatting',
  'make',
  'story',
  'hard',
  'follow',
  'die',
  'hard',
  'like',
  'want',
  'botherits',
  'sad',
  'thing',
  'good',
  'book',
  'spoiled',
  'formatting',
  'but',
  'fault',
  'author',
  'story',
  'this',
  'good',
  'book',
  'energy',
  'read',
  'itive',
  'also',
  'emailed',
  'author'],
 ['we',
  'already',
  'owned',
  'book',
  'my',
  'spouse',
  'forgot',
  'already',
  'part',
  'library',
  'book',
  'it',
  'unfortunate',
  'amazon',
  'safeguard',
  'place',
  'notify',
  'book',
  'already',
  'library'],
 ['cool',
  'forgot',
  'request',
  'rate',
  'came',
  'make',
  'mine',
  'unreliable',
  'rating',
  'doubt',
  'change',
  'rotation',
  'planet',
  'enough',
  'measure'],
 ['this',
  'short',
  'short',
  'story',
  'its',
  'basically',
  'scene',
  'party',
  

# Training Word2Vec model on Independent Features

In [120]:
from gensim.models import Word2Vec

model_word2vec = Word2Vec(sentences=X_train_words)

In [124]:
model_word2vec.wv.index_to_key

['book',
 'story',
 'read',
 'the',
 'one',
 'character',
 'like',
 'good',
 'this',
 'would',
 'really',
 'love',
 'it',
 'get',
 'time',
 'author',
 'reading',
 'series',
 'much',
 'first',
 'well',
 'even',
 'didnt',
 'short',
 'could',
 'know',
 'way',
 'sex',
 'great',
 'make',
 'little',
 'dont',
 'want',
 'two',
 'thing',
 'think',
 'he',
 'plot',
 'find',
 'also',
 'romance',
 'end',
 'there',
 'life',
 'im',
 'see',
 'go',
 'enjoyed',
 'scene',
 'never',
 'take',
 'written',
 'kindle',
 'many',
 'but',
 'say',
 'woman',
 'thought',
 'she',
 'work',
 'found',
 'lot',
 'and',
 'bit',
 'going',
 'interesting',
 'novel',
 'liked',
 'give',
 'loved',
 'year',
 'writing',
 'feel',
 'better',
 'come',
 'another',
 'got',
 'hot',
 'man',
 'though',
 'enough',
 'still',
 'if',
 'back',
 'review',
 'people',
 'made',
 'something',
 'cant',
 'part',
 'friend',
 'star',
 'reader',
 'page',
 'bad',
 'keep',
 'free',
 'need',
 'its',
 'new',
 'doesnt',
 'wasnt',
 'recommend',
 'enjoy',
 'wo

Function creation AvgWord2Vec

In [127]:
import numpy as np

def avgword2vec(doc):
    vectors = [model_word2vec.wv[word] for word in doc if word in model_word2vec.wv.index_to_key]
    if len(vectors) == 0:
        return np.zeros(model_word2vec.vector_size)
    return np.mean(vectors, axis=0)

In [128]:
from tqdm import tqdm

X_training_features = []
X_test_features = []

for i in tqdm(X_train_words):
    X_training_features.append(avgword2vec(i))

for i in tqdm(X_test_words):
    X_test_features.append(avgword2vec(i))

X_train_features = np.array(X_training_features)
X_test_features = np.array(X_test_features)

100%|██████████| 2400/2400 [00:09<00:00, 248.83it/s]


In [129]:
print(X_train_features.shape)
print(X_test_features.shape)

(9600, 100)
(2400, 100)


# Random Forest Model Training and Prediction

In [130]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier()

rf_model.fit(X_train_features, y_train)

RandomForestClassifier()

In [138]:
y_pred_rf = rf_model.predict(X_test_features)

In [135]:
!pip install xgboost

In [136]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier()

xgb_model.fit(X_train_features, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [137]:
y_pred_xgb = xgb_model.predict(X_test_features)

# Performance Metrics of Word2Vec

In [143]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("RandomForest Model on Word2Vec METRICS")
print("\nWord2Vec Accuracy:\n", accuracy_score(y_test, y_pred_rf))
print("\nWord2Vec Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("\nWord2Vec Classification Report:\n", classification_report(y_test, y_pred_rf))


print("\n\nXGBoost on Word2Vec METRICS")
print("\nWord2Vec Accuracy:\n", accuracy_score(y_test, y_pred_xgb))
print("\nWord2Vec Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("\nWord2Vec Classification Report:\n", classification_report(y_test, y_pred_xgb))

RandomForest Model on Word2Vec METRICS

Word2Vec Accuracy:
 0.7554166666666666

Word2Vec Confusion Matrix:
 [[ 421  382]
 [ 205 1392]]

Word2Vec Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.52      0.59       803
           1       0.78      0.87      0.83      1597

    accuracy                           0.76      2400
   macro avg       0.73      0.70      0.71      2400
weighted avg       0.75      0.76      0.75      2400



XGBoost on Word2Vec METRICS

Word2Vec Accuracy:
 0.7591666666666667

Word2Vec Confusion Matrix:
 [[ 458  345]
 [ 233 1364]]

Word2Vec Classification Report:
               precision    recall  f1-score   support

           0       0.66      0.57      0.61       803
           1       0.80      0.85      0.83      1597

    accuracy                           0.76      2400
   macro avg       0.73      0.71      0.72      2400
weighted avg       0.75      0.76      0.75      2400

